# PhoBERT Multi-Task with SMART
2-head training with SMART regularization

## 1. Setup

In [1]:
!pip install transformers accelerate py_vncorenlp -q

import os
import py_vncorenlp

os.makedirs('/kaggle/working/vncorenlp', exist_ok=True)
py_vncorenlp.download_model(save_dir='/kaggle/working/vncorenlp')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.8 MB/s eta 0:00:00


--2026-01-11 04:11:40--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 3.88M 7s
    50K .......... .......... .......... .......... ..........  0% 4.70M 6s
   100K .......... .......... .......... .......... ..........  0% 24.5M 4s
   150K .......... .......... .......... .......... ..........  0% 20.1M 4s
   200K .......... .......... .......... .......... ..........  0% 32.9M 3s
   250K .......... .......... .......... .......... ..........  1% 8.11M 3s
   300K .......... .......... .......... .......... ..........  1% 27.3M 3s
   350K ..

## 2. Configuration

In [ ]:
import os, random, numpy as np, pandas as pd
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

@dataclass
class Config:
    train_path: str = "/kaggle/input/neu-esc/train_set.csv"
    val_path: str = "/kaggle/input/neu-esc/val_set.csv"
    test_path: str = "/kaggle/input/neu-esc/test_set.csv"
    checkpoint_path: str = "/kaggle/working/phobert_smart_best.pt"
    resume_checkpoint: str = ""
    
    model_name: str = "vinai/phobert-base-v2"
    max_len: int = 256
    dropout: float = 0.3
    
    num_sentiment_classes: int = 4
    num_topic_classes: int = 10
    sentiment_labels: tuple = ("Neutral", "Positive", "Negative", "Toxic")
    topic_labels: tuple = ("Spam", "News", "Academic", "Other", "Service", "Jobs", "Personal", "Social", "Help", "Club")
    
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    epochs: int = 10
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    backbone_lr: float = 1e-5
    head_lr: float = 5e-5
    
    label_smoothing: float = 0.1
    early_stopping_patience: int = 3
    
    smart_epsilon: float = 1e-5
    smart_alpha: float = 0.02
    smart_steps: int = 1
    smart_lambda_sent: float = 1.0
    smart_lambda_topic: float = 1.0
    bregman_lambda: float = 0.01
    
    seed: int = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

config = Config()
set_seed(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 3. Load Data

In [3]:
import py_vncorenlp
tqdm.pandas()

rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir='/kaggle/working/vncorenlp')

def segment_text(text):
    try:
        if not text or pd.isna(text): return ""
        result = rdrsegmenter.word_segment(str(text))
        if isinstance(result, list):
            return " ".join([" ".join(s) if isinstance(s, list) else s for s in result])
        return result
    except: return str(text)

def load_data(config):
    train_df = pd.read_csv(config.train_path)
    val_df = pd.read_csv(config.val_path)
    test_df = pd.read_csv(config.test_path)
    for df in [train_df, val_df, test_df]:
        df['sentiment'] = df['sentiment'].astype(int)
        df['classification'] = df['classification'].astype(int)
    
    train_df['text'] = train_df['text'].progress_apply(segment_text)
    val_df['text'] = val_df['text'].progress_apply(segment_text)
    test_df['text'] = test_df['text'].progress_apply(segment_text)
    
    return train_df, val_df, test_df

def compute_class_weights(labels, num_classes):
    counts = np.maximum(np.bincount(labels, minlength=num_classes), 1)
    weights = len(labels) / (num_classes * counts)
    return torch.tensor(weights / weights.sum() * num_classes, dtype=torch.float32)

train_df, val_df, test_df = load_data(config)
sentiment_weights = compute_class_weights(train_df['sentiment'].values, config.num_sentiment_classes)
topic_weights = compute_class_weights(train_df['classification'].values, config.num_topic_classes)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

2026-01-11 04:11:59 INFO  WordSegmenter:24 - Loading Word Segmentation model


  0%|          | 0/23048 [00:00<?, ?it/s]

  0%|          | 0/3305 [00:00<?, ?it/s]

  0%|          | 0/6613 [00:00<?, ?it/s]

Train: 23048, Val: 3305, Test: 6613


## 4. Dataset

In [4]:
class NEUESCDataset(Dataset):
    def __init__(self, texts, sentiments, topics, tokenizer, max_len):
        self.texts, self.sentiments, self.topics = texts, sentiments, topics
        self.tokenizer, self.max_len = tokenizer, max_len
    
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(self.sentiments[idx], dtype=torch.long),
            'topic_label': torch.tensor(self.topics[idx], dtype=torch.long)
        }

## 5. Model

In [5]:
class PhoBERTMultiTaskSMART(nn.Module):
    """2-head model with SMART support (NEU-ESC paper architecture)."""
    def __init__(self, model_name, num_sent=4, num_topic=10, dropout=0.1):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        hidden = self.phobert.config.hidden_size
        
        self.shared_linear = nn.Linear(hidden, hidden)
        self.layer_norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(dropout)
        self.sent_clf = nn.Linear(hidden, num_sent)
        self.topic_clf = nn.Linear(hidden, num_topic)
    
    def get_embeddings(self, input_ids):
        return self.phobert.embeddings(input_ids)
    
    def forward_from_embeddings(self, embeddings, attention_mask):
        ext_mask = (1.0 - attention_mask[:, None, None, :]) * -10000.0
        seq_out = self.phobert.encoder(embeddings, attention_mask=ext_mask).last_hidden_state
        cls_out = seq_out[:, 0, :]
        cls_proj = self.dropout(self.layer_norm(self.shared_linear(cls_out)))
        return self.sent_clf(cls_proj), self.topic_clf(cls_proj)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        cls_out = outputs.last_hidden_state[:, 0, :]
        cls_proj = self.dropout(self.layer_norm(self.shared_linear(cls_out)))
        return self.sent_clf(cls_proj), self.topic_clf(cls_proj)

## 6. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight, self.gamma, self.label_smoothing = weight, gamma, label_smoothing
    
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none', label_smoothing=self.label_smoothing)
        p_t = F.softmax(inputs, dim=-1).gather(1, targets.unsqueeze(1)).squeeze(1)
        return (((1 - p_t) ** self.gamma) * ce).mean()


class SMARTLoss(nn.Module):
    """SMART regularization for classification heads."""
    def __init__(self, epsilon=1e-5, alpha=0.02, steps=1):
        super().__init__()
        self.epsilon, self.alpha, self.steps = epsilon, alpha, steps
    
    def forward(self, model, embeddings, attention_mask, sent_logits_clean, topic_logits_clean):
        delta = torch.zeros_like(embeddings).uniform_(-self.epsilon, self.epsilon)
        delta.requires_grad = True
        
        for _ in range(self.steps):
            sent_adv, topic_adv = model.forward_from_embeddings(embeddings + delta, attention_mask)
            kl = F.kl_div(F.log_softmax(sent_adv, -1), F.softmax(sent_logits_clean.detach(), -1), reduction='batchmean') + \
                 F.kl_div(F.log_softmax(topic_adv, -1), F.softmax(topic_logits_clean.detach(), -1), reduction='batchmean')
            kl.backward(retain_graph=True)
            delta = torch.clamp(delta + self.alpha * delta.grad.detach().sign(), -self.epsilon, self.epsilon).detach()
            delta.requires_grad = True
        
        sent_adv, topic_adv = model.forward_from_embeddings(embeddings + delta, attention_mask)
        R_sent = F.kl_div(F.log_softmax(sent_adv, -1), F.softmax(sent_logits_clean.detach(), -1), reduction='batchmean')
        R_topic = F.kl_div(F.log_softmax(topic_adv, -1), F.softmax(topic_logits_clean.detach(), -1), reduction='batchmean')
        return R_sent, R_topic

class BregmanProximalLoss:

    def __init__(self, pretrained_weights, lambda_prox=0.01):
        self.pretrained_weights = pretrained_weights
        self.lambda_prox = lambda_prox

    def compute_loss(self, model):
        """Compute L2 distance from pre-trained weights."""
        prox_loss = 0.0
        for name, param in model.named_parameters():
            if name in self.pretrained_weights:
                prox_loss += torch.norm(param - self.pretrained_weights[name]) ** 2
        return self.lambda_prox * prox_loss


class MultiTaskLoss(nn.Module):
    def __init__(self, sent_weights, topic_weights, label_smoothing=0.0, device=None):
        super().__init__()
        self.sent_loss_fn = FocalLoss(sent_weights.to(device), label_smoothing=label_smoothing)
        self.topic_loss_fn = FocalLoss(topic_weights.to(device), label_smoothing=label_smoothing)
    
    def forward(self, sent_logits, topic_logits, sent_labels, topic_labels):
        return self.sent_loss_fn(sent_logits, sent_labels), self.topic_loss_fn(topic_logits, topic_labels)

## 7. Training Functions

In [ ]:
def train_epoch(model, loader, loss_fn, smart_fn, optimizer, scheduler, config):
    model.train()
    total_loss, total_sent, total_topic, total_smart = 0, 0, 0, 0
    optimizer.zero_grad()
    
    for idx, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
        sent_y, topic_y = batch['sentiment_label'].to(device), batch['topic_label'].to(device)
        
        emb = model.get_embeddings(ids)
        sent_logits, topic_logits = model.forward_from_embeddings(emb, mask)
        
        L_sent, L_topic = loss_fn(sent_logits, topic_logits, sent_y, topic_y)
        R_sent, R_topic = smart_fn(model, emb.detach(), mask, sent_logits, topic_logits)
        
        loss = (L_sent + config.smart_lambda_sent * R_sent) + (L_topic + config.smart_lambda_topic * R_topic)
        
        L_bregman = bregman_fn.compute_loss(model)
        
        (loss / config.gradient_accumulation_steps).backward()
        
        total_loss += loss.item()
        total_sent += L_sent.item()
        total_topic += L_topic.item()
        total_smart += (R_sent.item() + R_topic.item() + L_bregman.item())
        
        if (idx + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
    
    n = len(loader)
    return {'loss': total_loss/n, 'sent': total_sent/n, 'topic': total_topic/n, 'smart': total_smart/n}


@torch.no_grad()
def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    sent_preds, sent_labels, topic_preds, topic_labels = [], [], [], []
    
    for batch in tqdm(loader, desc="Eval", leave=False):
        ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
        sent_y, topic_y = batch['sentiment_label'].to(device), batch['topic_label'].to(device)
        
        sent_logits, topic_logits = model(ids, mask)
        L_sent, L_topic = loss_fn(sent_logits, topic_logits, sent_y, topic_y)
        total_loss += (L_sent + L_topic).item()
        
        sent_preds.extend(sent_logits.argmax(1).cpu().numpy())
        sent_labels.extend(sent_y.cpu().numpy())
        topic_preds.extend(topic_logits.argmax(1).cpu().numpy())
        topic_labels.extend(topic_y.cpu().numpy())
    
    return {'loss': total_loss/len(loader), 'sp': np.array(sent_preds), 'sl': np.array(sent_labels),
            'tp': np.array(topic_preds), 'tl': np.array(topic_labels)}


def compute_metrics(sp, sl, tp, tl, config, report=False):
    m = {'s_acc': accuracy_score(sl, sp), 's_f1': f1_score(sl, sp, average='macro', zero_division=0),
         't_acc': accuracy_score(tl, tp), 't_f1': f1_score(tl, tp, average='macro', zero_division=0)}
    m['combined'] = (m['s_f1'] + m['t_f1']) / 2
    if report:
        print("\nSentiment:")
        print(classification_report(sl, sp, labels=list(range(config.num_sentiment_classes)), target_names=list(config.sentiment_labels), zero_division=0))
        print("\nTopic:")
        print(classification_report(tl, tp, labels=list(range(config.num_topic_classes)), target_names=list(config.topic_labels), zero_division=0))
    return m

## 8. Training Loop

In [ ]:
def train_model(config):
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    
    train_ds = NEUESCDataset(train_df['text'].tolist(), train_df['sentiment'].tolist(), train_df['classification'].tolist(), tokenizer, config.max_len)
    val_ds = NEUESCDataset(val_df['text'].tolist(), val_df['sentiment'].tolist(), val_df['classification'].tolist(), tokenizer, config.max_len)
    train_ld = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=2)
    val_ld = DataLoader(val_ds, batch_size=config.batch_size, num_workers=2)
    
    model = PhoBERTMultiTaskSMART(config.model_name, config.num_sentiment_classes, config.num_topic_classes, config.dropout).to(device)
    pretrained_backbone_weights = {}
    for name, param in model.named_parameters():
        if 'phobert' in name:  
            pretrained_backbone_weights[name] = param.clone().detach()
    loss_fn = MultiTaskLoss(sentiment_weights, topic_weights, config.label_smoothing, device)
    smart_fn = SMARTLoss(config.smart_epsilon, config.smart_alpha, config.smart_steps)
    bregman_fn = BregmanProximalLoss(pretrained_backbone_weights, config.bregman_lambda)
    
    opt_params = [
        {'params': model.phobert.parameters(), 'lr': config.backbone_lr},
        {'params': model.shared_linear.parameters(), 'lr': config.head_lr},
        {'params': model.layer_norm.parameters(), 'lr': config.head_lr, 'weight_decay': 0.0},
        {'params': model.sent_clf.parameters(), 'lr': config.head_lr},
        {'params': model.topic_clf.parameters(), 'lr': config.head_lr}
    ]
    optimizer = AdamW(opt_params, weight_decay=config.weight_decay)
    total_steps = len(train_ld) * config.epochs // config.gradient_accumulation_steps
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * config.warmup_ratio), total_steps)
    
    start_epoch, best_f1, patience = 0, 0.0, 0
    
    if config.resume_checkpoint and os.path.exists(config.resume_checkpoint):
        ckpt = torch.load(config.resume_checkpoint, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt.get('scheduler_state_dict', scheduler.state_dict()))
        start_epoch, best_f1 = ckpt.get('epoch', 0) + 1, ckpt.get('best_f1', 0.0)
        pretrained_backbone_weights = {}
        for name, param in model.named_parameters():
            if 'phobert' in name:
                pretrained_backbone_weights[name] = param.clone().detach()
        bregman_fn = BregmanProximalLoss(pretrained_backbone_weights, config.bregman_lambda)
        print(f"Resumed from epoch {start_epoch}, best F1: {best_f1:.4f}")
    
    for ep in range(start_epoch, config.epochs):
        print(f"\nEpoch {ep+1}/{config.epochs}")
        tr = train_epoch(model, train_ld, loss_fn, smart_fn, optimizer, scheduler, config)
        print(f"Train - Loss: {tr['loss']:.4f} (sent: {tr['sent']:.3f}, topic: {tr['topic']:.3f}, smart: {tr['smart']:.3f})")
        
        vr = eval_epoch(model, val_ld, loss_fn)
        m = compute_metrics(vr['sp'], vr['sl'], vr['tp'], vr['tl'], config)
        print(f"Val - Loss: {vr['loss']:.4f}, Sent F1: {m['s_f1']:.4f}, Topic F1: {m['t_f1']:.4f}, Combined: {m['combined']:.4f}")
        
        if m['combined'] > best_f1:
            best_f1, patience = m['combined'], 0
            torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(), 'epoch': ep, 'best_f1': best_f1, 'metrics': m}, config.checkpoint_path)
            print(f">> Saved best model (F1: {best_f1:.4f})")
        else:
            patience += 1
            print(f">> No improvement ({patience}/{config.early_stopping_patience})")
            if patience >= config.early_stopping_patience:
                print("Early stopping!")
                break
    
    print(f"\nTraining complete. Best F1: {best_f1:.4f}")
    return model, tokenizer

## 9. Run Training

In [9]:
model, tokenizer = train_model(config)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-01-11 04:12:23.171193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768104743.352956      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768104743.410248      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768104743.848879      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768104743.848916      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768104743.848920      24 computation_placer.cc:177] computation placer alr

...

Val - Loss: 0.3499, Sent F1: 0.7330, Topic F1: 0.6398, Combined: 0.6864
>> Saved best model (F1: 0.6864)

Epoch 10/10


Train:   0%|          | 0/1441 [00:00<?, ?it/s]

Train - Loss: 0.2387 (sent: 0.077, topic: 0.125, smart: 0.036)


Eval:   0%|          | 0/207 [00:00<?, ?it/s]

Val - Loss: 0.3489, Sent F1: 0.7264, Topic F1: 0.6401, Combined: 0.6833
>> No improvement (1/3)

Training complete. Best F1: 0.6864


## 10. Evaluate on Test Set

In [ ]:
ckpt = torch.load(config.checkpoint_path, map_location=device)
model = PhoBERTMultiTaskSMART(config.model_name, config.num_sentiment_classes, config.num_topic_classes, config.dropout).to(device)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model (Val F1: {ckpt['best_f1']:.4f})")

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
test_ds = NEUESCDataset(test_df['text'].tolist(), test_df['sentiment'].tolist(), test_df['classification'].tolist(), tokenizer, config.max_len)
test_ld = DataLoader(test_ds, batch_size=config.batch_size, num_workers=2)

loss_fn = MultiTaskLoss(sentiment_weights, topic_weights, config.label_smoothing, device)
tr = eval_epoch(model, test_ld, loss_fn)
m = compute_metrics(tr['sp'], tr['sl'], tr['tp'], tr['tl'], config, report=True)

print(f"\nTest Results:")
print(f"Sentiment - Acc: {m['s_acc']*100:.2f}%, F1: {m['s_f1']*100:.2f}%")
print(f"Topic     - Acc: {m['t_acc']*100:.2f}%, F1: {m['t_f1']*100:.2f}%")
print(f"Combined F1: {m['combined']*100:.2f}%")

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded best model (Val F1: 0.6864)


Eval:   0%|          | 0/414 [00:00<?, ?it/s]


Sentiment:
              precision    recall  f1-score   support

     Neutral       0.90      0.85      0.87      4558
    Positive       0.64      0.72      0.68       834
    Negative       0.65      0.72      0.68      1046
       Toxic       0.71      0.87      0.79       175

    accuracy                           0.81      6613
   macro avg       0.73      0.79      0.76      6613
weighted avg       0.82      0.81      0.82      6613


Topic:
              precision    recall  f1-score   support

        Spam       0.50      0.62      0.55        84
        News       0.49      0.58      0.53       183
    Academic       0.84      0.83      0.83      2104
       Other       0.88      0.82      0.85      2882
     Service       0.62      0.65      0.64       474
        Jobs       0.58      0.74      0.65       163
    Personal       0.50      0.50      0.50       298
      Social       0.53      0.56      0.54       155
        Help       0.40      0.59      0.48       137
    